# Topic 37 — Transformers
### Theory → positional encoding → a full Transformer encoder block, from scratch and in PyTorch.

A Transformer assembles the pieces from Topic 36 (multi-head self-attention) with a few more
components into one repeatable **block**, stacked several times:

```text
Input embeddings + positional encoding
        |
   Multi-Head Self-Attention
        |
   Add & Norm (residual connection + layer norm)
        |
   Feed-Forward Network
        |
   Add & Norm
        |
      Output  (repeat this whole block N times = "N layers")
```

Unlike RNNs/LSTMs (Topics 33-34), a Transformer has NO recurrence — it processes the whole
sequence in parallel, which is why it needs positional encoding to know word order at all.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)

## 1. Positional encoding — injecting word order

Since self-attention treats the input as an unordered SET of vectors (no notion of "1st word,
2nd word..."), we must explicitly add position information. The original Transformer paper uses
sine/cosine functions of different frequencies:

```text
PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
```

This gives each position a unique pattern, and — usefully — the relative distance between any two
positions can be recovered from their encodings via simple linear operations.

In [ ]:
def positional_encoding(seq_len, d_model):
    pos = np.arange(seq_len)[:, np.newaxis]
    i = np.arange(d_model)[np.newaxis, :]
    angle_rates = 1 / np.power(10000, (2 * (i // 2)) / d_model)
    angles = pos * angle_rates

    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(angles[:, 0::2])   # even indices: sine
    pe[:, 1::2] = np.cos(angles[:, 1::2])   # odd indices: cosine
    return pe

pe = positional_encoding(seq_len=50, d_model=64)

plt.figure(figsize=(8, 5))
plt.imshow(pe.T, cmap="RdBu", aspect="auto")
plt.xlabel("position in sequence")
plt.ylabel("embedding dimension")
plt.title("Positional encoding pattern")
plt.colorbar()
plt.show()
# Each row is a different sine/cosine wave at a different frequency -- together, every POSITION
# gets a unique combination of values across all dimensions.

In [ ]:
# Positional encoding gets ADDED (not concatenated) to the word embeddings
d_model = 8
seq_len = 5
word_embeddings = torch.randn(seq_len, d_model)
pe_small = torch.tensor(positional_encoding(seq_len, d_model), dtype=torch.float32)

combined = word_embeddings + pe_small
print("word embeddings shape:", word_embeddings.shape)
print("positional encoding shape:", pe_small.shape)
print("combined shape (same as input):", combined.shape)
# The model receives position-aware embeddings, but the SHAPE never changes -- position info
# is blended directly into the same vectors, not stored as a separate signal.

## 2. Residual connections & layer normalization

- **Residual connection**: `output = SubLayer(x) + x` — adding the ORIGINAL input back after a
  sub-layer (attention or feed-forward). Helps gradients flow through many stacked layers without
  vanishing (Topic 33's problem, addressed differently here than LSTM's gating).
- **Layer normalization**: normalizes across the FEATURE dimension for each individual sample
  (unlike BatchNorm, Topic 31, which normalizes across the batch) — stabilizes training,
  especially important since Transformers stack many layers deep.

In [ ]:
layer_norm = nn.LayerNorm(d_model)

x = torch.randn(seq_len, d_model) * 5 + 3   # some arbitrarily-scaled input
normalized = layer_norm(x)

print("before layer norm -- mean:", x.mean().item(), " std:", x.std().item())
print("after layer norm -- mean per row ~0:", normalized.mean(dim=-1))
print("after layer norm -- std per row ~1:", normalized.std(dim=-1))

# Residual connection example
sub_layer_output = torch.randn(seq_len, d_model) * 0.1   # imagine this came from attention
residual_output = sub_layer_output + x   # add the original input back
print("\nresidual connection preserves the original signal even if sub_layer_output is small/noisy")

## 3. Feed-forward network

After attention mixes information ACROSS positions, a simple 2-layer feed-forward network
(applied identically, independently, to EACH position) adds further per-position processing —
usually expand-then-contract: `d_model -> d_ff (bigger) -> d_model`.

In [ ]:
feed_forward = nn.Sequential(
    nn.Linear(d_model, d_model * 4),   # expand
    nn.ReLU(),
    nn.Linear(d_model * 4, d_model),   # contract back
)

ff_output = feed_forward(x)
print("feed-forward output shape (unchanged):", ff_output.shape)

## 4. Assembling one full Transformer encoder block

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, attn_weights = self.self_attn(x, x, x)        # self-attention (Topic 36)
        x = self.norm1(x + attn_out)                              # residual + layer norm
        ff_out = self.ff(x)                                        # feed-forward
        x = self.norm2(x + ff_out)                                 # residual + layer norm
        return x, attn_weights

block = TransformerEncoderBlock(d_model=16, n_heads=4, d_ff=64)
X_input = torch.randn(1, 6, 16)   # (batch, seq_len, d_model)

output, attn_weights = block(X_input)
print("input shape:", X_input.shape)
print("output shape (unchanged -- can stack blocks!):", output.shape)
print("attention weights shape:", attn_weights.shape)

## 5. Stacking multiple blocks — a mini Transformer encoder

In [ ]:
class MiniTransformerEncoder(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, n_layers):
        super().__init__()
        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)
        ])

    def forward(self, x):
        for block in self.blocks:
            x, _ = block(x)
        return x

encoder = MiniTransformerEncoder(d_model=16, n_heads=4, d_ff=64, n_layers=3)
final_output = encoder(X_input)
print("output after 3 stacked encoder blocks:", final_output.shape)
print("total parameters:", sum(p.numel() for p in encoder.parameters()))

## 6. Masked attention — brief explanation

**Masked (causal) self-attention** prevents a position from attending to FUTURE positions —
essential for the DECODER side of a Transformer (e.g. text generation), where predicting word `t`
must only use words `1...t-1`, never peek ahead at the answer. Implemented by setting future
positions' attention scores to `-infinity` before the softmax, so they get weight ≈ 0.

In [ ]:
def create_causal_mask(seq_len):
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
    return mask   # True = "block this position"

mask = create_causal_mask(5)
print("causal mask (True = blocked / cannot attend to this position):")
print(mask)

plt.figure(figsize=(4, 4))
plt.imshow(mask.numpy(), cmap="gray_r")
plt.title("Causal mask -- each row can only see itself + earlier positions")
plt.xlabel("attending TO position"); plt.ylabel("attending FROM position")
plt.show()

# Using it with nn.MultiheadAttention:
masked_attn = nn.MultiheadAttention(16, 4, batch_first=True)
out, weights = masked_attn(X_input, X_input, X_input, attn_mask=mask)
print("\nwith causal mask, position 0's attention weights (should only be nonzero at position 0):")
print(weights[0, 0].detach().numpy().round(3))

## 7. Encoder vs decoder — one-paragraph summary

- **Encoder** (what we built above): reads the WHOLE input at once, bidirectionally — every
  position can attend to every other position. Used for understanding tasks (classification,
  BERT-style models — Topic 38).
- **Decoder**: generates output one token at a time, using CAUSAL (masked) self-attention on what
  it's generated so far, plus (in encoder-decoder architectures like translation) cross-attention
  over the encoder's output. Used for generation tasks (GPT-style models).

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Change n_layers to 1 and to 6 in MiniTransformerEncoder -- how does total parameter count scale?
# 2. Print positional_encoding(seq_len=10, d_model=4) directly and manually verify a couple of
#    sin/cos values against the formula.
# 3. Remove the residual connections in TransformerEncoderBlock (just use attn_out and ff_out
#    directly, no "+x") and see if training becomes unstable on a toy task -- this demonstrates
#    WHY residuals matter, not just states it.
# 4. In one sentence: why does a Transformer encoder process a whole sequence in PARALLEL while
#    an LSTM (Topic 34) must process it SEQUENTIALLY, one timestep at a time -- and why does that
#    make Transformers much faster to train on modern GPUs?

---
### Next up: **Topic 38 — BERT-style models** (pretraining, fine-tuning, using Hugging Face transformers).

Say "next" when you're ready.